# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR² [“Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells”](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via its Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset via the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
dataset_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(dataset_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review all available record sets, their fields, and associated `@id`s.

We'll inspect the structure using the Croissant schema and the `mlcroissant` API.

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets

print(f"Available record sets (by @id):\n")
for rs in record_sets:
    print(f"  - @id: {rs.id}")
    print(f"    name: {rs.name}")
    print(f"    description: {rs.description if hasattr(rs, 'description') else '-'}")
    print(f"    Fields:")
    for field in rs.fields:
        print(f"      - @id: {field.id}; name: {field.name}; dataType: {field.data_type}")
    print()

## 3. Data Extraction
We'll load tabular data from a primary record set into a DataFrame for analysis. Use the record set and field `@id`s as shown in the overview above. 

_**Note:** This dataset may contain one or more record sets. We'll demonstrate extracting the main one for further analysis._

In [ ]:
# Choose a record set for extraction (by @id)
# See previous cell for available record sets and change the chosen_record_set_id as needed.

# If the dataset lists no record sets, you can iterate all files as a fallback (some datasets only have files available)
if len(dataset.record_sets) == 0:
    print("No explicit record sets found. Listing DataFrames from provided files/distributions...")
    dataframes = {}
    for i, dist in enumerate(dataset.metadata.distribution):
        df = dataset.read_distribution(distribution=dist.id)
        dataframes[dist.id] = df
        print(f"Loaded DataFrame from distribution @id='{dist.id}': columns={df.columns.tolist()}")
    # Pick the main DataFrame (assumes 1st)
    if dataframes:
        main_dist_id = list(dataframes.keys())[0]
        main_df = dataframes[main_dist_id]
    else:
        raise ValueError("No data files found in the dataset.")
else:
    # If record sets are present
    chosen_record_set = dataset.record_sets[0]
    chosen_record_set_id = chosen_record_set.id
    print(f"Extracting data from record set: {chosen_record_set_id} ({chosen_record_set.name})")
    records = list(dataset.records(record_set=chosen_record_set_id))
    main_df = pd.DataFrame(records)
    print(f"Loaded {len(main_df)} records with columns: {main_df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. We must reference all columns by their `@id`, as per best practices for Croissant datasets.

_You may need to consult the previous overview step for the exact `@id` of the fields/columns to use._

In [ ]:
# Example: Pick a numeric protein abundance field for filtering and normalization.

# Inspect column names (these often mirror field @id but may be simplified in the DataFrame)
print("Available DataFrame columns (may or may not match field @id):")
print(main_df.columns.tolist())

# Try to select a field likely to be numeric. Adjust as needed based on available columns.
# Common names: 'MW', 'coverage', 'abundance', etc. Simulate as if the column is '@id: cr:MW'

numeric_field_id = None
for col in main_df.columns:
    if 'mw' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Fallback: pick the first numeric column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

if numeric_field_id is not None:
    print(f"Using numeric field: {numeric_field_id}\n")
else:
    raise RuntimeError("Could not identify a numeric field for EDA.")

# Filter: remove missing and outlier records with value > threshold (e.g., 10)
threshold = 10
filtered_df = main_df[main_df[numeric_field_id].astype(float) > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold} (count={len(filtered_df)}):")
print(filtered_df.head())

# Normalize this field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
) / filtered_df[numeric_field_id].astype(float).std()

print(f"\nNormalized ({numeric_field_id}):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if one is available (e.g. by protein type/description)
group_field = None
for col in main_df.columns:
    if col != numeric_field_id and main_df[col].nunique() > 1 and main_df[col].nunique() < len(main_df) // 2:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nGrouped mean {numeric_field_id} by '{group_field}':")
    print(grouped_df.head())
else:
    print("\nNo suitable categorical group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of the normalized numeric field, or the mean by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (z-score)")
plt.tight_layout()
plt.show()

# If we have a group field, plot group means
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(8, 4))
    grouped_df.plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR² dataset: *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*. 

- We loaded the dataset's schema and metadata directly via the public Croissant schema URL.
- We inspected available record sets and fields, then loaded tabular data into pandas.
- We performed basic filtering, normalization, and (if available) grouping by key attributes for exploratory analysis.
- We visualized distributions to identify main patterns.

**You can now customize further analysis steps to your research/application needs!**